### 1. Initialization and read

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

In [0]:
# Read Bronze tables
df = spark.table("catalogue_project1.bronze.crm_cust_info")

### 2. Transformations
2.1 Dealing with NULLs and zeroes

In [0]:
df = df.filter(col("cst_id").isNotNull())

2.2 Trimming (unwanted spaces)

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

2.3 Standarization

In [0]:
df = (
    df.withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .otherwise("n/a")
    )
    .withColumn("cst_gndr",
                F.when(F.upper(F.col("cst_gndr")) == "M", "Male")
                 .when(F.upper(F.col("cst_gndr")) == "F", "Female")
                 .otherwise("n/a"))
    )


2.4 Rename columns

In [0]:
RENAME_CONFIG = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "create_date"
}
for old_name, new_name in RENAME_CONFIG.items():
    df = df.withColumnRenamed(old_name, new_name)

* Intermediate step: check the changes applied correctly.

In [0]:
df.limit(10).display()

### 3. Write into the Silver delta table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("catalogue_project1.silver.crm_customers")

* Check that everything went well

In [0]:
%sql
SELECT * FROM catalogue_project1.silver.crm_customers LIMIT 100